# 从时域到频域的“透视眼”：NR 工程师的 FFT 与加窗实战指南

**作者：** [余东海]  
**领域：** 5G NR 基带处理 / 信号处理  

### 导言
在 5G NR (New Radio) 的世界里，信号在空气中是杂乱无章的电磁波。作为软件工程师，我们最核心的任务之一就是利用 **FFT (快速傅里叶变换)** 将这些时域采样点还原为频域的子载波。

但在实际工程中，信号总是被“截断”处理的。本文将带你通过 Python 实验，直观感受 **FFT 变换**、**频谱泄露**、**加窗 (Windowing)** 以及它们对 **16-QAM 星座图** 的真实影响。

## 1. 数学基石：DFT 与 FFT
离散傅里叶变换 (DFT) 的定义如下：
$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j\frac{2\pi}{N}kn}$$

其中 $e^{-j\frac{2\pi}{N}kn}$ 被称为**旋转因子**。FFT 则是 DFT 的一种高效实现，将复杂度从 $O(N^2)$ 降到了 $O(N \log N)$。对于 NR 中高采样率（如 122.88 MHz）的信号处理，FFT 是实时性的保障。

## 2. 实验：模拟 16-QAM 信号截断与频谱泄露
我们将生成一段 16-QAM 信号。由于硬件缓冲区有限，我们只能截断其中的一小段进行 FFT。这种“截断”在数学上等同于给信号加了一个**矩形窗 (Rectangular Window)**，它是频谱泄露的元凶。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

# 设置全局绘图风格
plt.rcParams['font.sans-serif'] = ['SimHei'] # 如果是中文环境，确保字体显示正常
plt.rcParams['axes.unicode_minus'] = False


# ==========================================
# 1. 参数定义
# ==========================================
n_fft = 1024        # FFT 点数
fs = 30.72e6        # 采样率 (NR 标准 30.72MHz)
m_order = 16        # 16-QAM
truncate_len = 256  # 截断长度（观察窗效应）

# ==========================================
# 2. 生成 16-QAM 星座点与时域信号
# ==========================================
# 生成理想星座图
points = np.array([-3, -1, 1, 3])
mapping = np.array([r + 1j*c for r in points for c in points])
mapping /= np.sqrt(10) # 功率归一化

# 模拟 256 个时域采样点 (简化模型)
tx_symbols = np.random.choice(mapping, truncate_len)
# 添加少量高斯白噪声 (SNR 约 25dB)
noisy_signal = tx_symbols + (np.random.randn(truncate_len) + 1j*np.random.randn(truncate_len)) * 0.03

# ==========================================
# 3. 加窗处理 (对比组)
# ==========================================
# 汉宁窗：两端平滑衰减，减少截断处的突变
hanning_win = signal.windows.hann(truncate_len)
windowed_signal = noisy_signal * hanning_win

# ==========================================
# 4. 执行 FFT 并计算功率谱 (dB)
# ==========================================
def get_psd(sig, n):
    spectrum = np.fft.fft(sig, n)
    psd = 20 * np.log10(np.abs(np.fft.fftshift(spectrum)) + 1e-10)
    return psd

psd_rect = get_psd(noisy_signal, n_fft)
psd_hann = get_psd(windowed_signal, n_fft)
freq_axis = np.fft.fftshift(np.fft.fftfreq(n_fft, d=1/fs))

# ==========================================
# 5. 绘图展示
# ==========================================
plt.figure(figsize=(15, 10))

# 子图 1: 频域对比 (频谱泄露)
plt.subplot(2, 2, 1)
plt.plot(freq_axis/1e6, psd_rect, label='矩形窗 (不加窗)', alpha=0.7)
plt.plot(freq_axis/1e6, psd_hann, label='汉宁窗', color='orange', linewidth=2)
plt.title("频谱泄露对比 (Frequency Leakage)")
plt.xlabel("频率 (MHz)"); plt.ylabel("幅度 (dB)")
plt.grid(True, linestyle='--'); plt.legend()

# 子图 2: 时域窗函数形态
plt.subplot(2, 2, 2)
plt.plot(hanning_win, color='orange', label='Hanning Window')
plt.fill_between(range(truncate_len), hanning_win, alpha=0.2, color='orange')
plt.title("汉宁窗时域包络")
plt.grid(True); plt.legend()

# 子图 3: 原始星座图
plt.subplot(2, 2, 3)
plt.scatter(noisy_signal.real, noisy_signal.imag, s=15, alpha=0.6)
plt.title("星座图: 原始截断信号")
plt.axhline(0, color='black', lw=1); plt.axvline(0, color='black', lw=1)
plt.grid(True); plt.axis('equal')

# 子图 4: 加窗后的星座图
plt.subplot(2, 2, 4)
plt.scatter(windowed_signal.real, windowed_signal.imag, s=15, alpha=0.6, color='orange')
plt.title("星座图: 加窗后的变化")
plt.axhline(0, color='black', lw=1); plt.axvline(0, color='black', lw=1)
plt.grid(True); plt.axis('equal')

plt.tight_layout()
plt.show()

## 3. 深度分析：工程师的思考



### 频谱泄露 (Spectral Leakage)
观察左上角的频谱图，你会发现**蓝色曲线（矩形窗）**的旁瓣非常高。在 NR 系统中，这会导致能量蔓延到相邻的子载波，造成严重的 **ICI (子载波间干扰)**。而**橙色曲线（汉宁窗）**的旁瓣下降极快，保持了频带的“整洁”。

### 星座图的“陷阱”
观察下方的星座图对比：
1. **左侧（原始）**：星座点虽然受噪声影响略有散开，但依然保持了 16-QAM 的方格阵列。
2. **右侧（加窗后）**：星座点向原点方向收缩并变得模糊。
**这是为什么？** 因为窗函数在时域上压低了信号两端的振幅。
**结论：** 在 NR 接收机中，如果我们为了抑制干扰在前端加了窗，那么在解调星座图之前，必须进行**增益补偿或信道均衡**。否则，EVm (误差矢量幅度) 将会爆表，导致无法正确解码。

## 总结
作为软件工程师，实现 FFT 只是第一步。理解**窗函数导致的能量分布改变**，以及如何在**抑制干扰**和**保持星座图质量**之间寻找平衡，才是基带算法开发的精髓所在。